## Baseline TTS voices generation using Google Cloud TTS

files required:
- en_US_voices.csv --> list of voices to be generated
- prompts2.csv --> list of prompts to be synthesized

In [5]:
!pip install gcloud
!pip install google-cloud-texttospeech

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.4/454.4 kB 7.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for gcloud: filename=gcloud-0.18.3-py3-none-any.whl size=602925 sha256=17798822061564ebdb040eced2524a6df9bcf8d82ae7976505998caa49c16efd
  Stored in directory: /root/.cache/pip/wheels/7c/30/88/5017af921da3a33af785f0d0fd3e944b845bc62a445a2c2f69
Successfully built gcloud
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.5/183.5 kB 3.2 MB/s eta 0:00:00


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
%cd /content/drive/MyDrive/PhD expts/Voice Match/

/content/drive/MyDrive/PhD expts/Voice Match


In [12]:
from google.colab import auth
auth.authenticate_user()

MessageError: Error: credential propagation was unsuccessful

In [11]:
"""Synthesizes speech from the input string of text or ssml.
Make sure to be working in a virtual environment.

Note: ssml must be well-formed according to:
    https://www.w3.org/TR/speech-synthesis/
"""
from google.cloud import texttospeech
import csv
import os

file = open('en_US_voices.csv', "r", encoding='utf-8-sig')
v_list = csv.reader(file)

cwd = os.getcwd()

for row in v_list:

    v_name = ''.join(map(str,row))
    path = os.path.join(cwd, v_name)
  #  os.mkdir(path)

    # Instantiates a client
    client = texttospeech.TextToSpeechClient()
    # Build the voice request
    voice = texttospeech.VoiceSelectionParams(language_code="en-US", name = v_name)

    #Open csv containing speech prompts
    file = open('prompts2.csv', "r", encoding='utf-8-sig')
    csvreader = csv.reader(file)
    i = 0

    for row in csvreader:
       #config
       audio_config = texttospeech.AudioConfig(
       audio_encoding=texttospeech.AudioEncoding.LINEAR16,
       sample_rate_hertz= 16000)
       # Set the text input to be synthesized
       in_text = ''.join(row)
       print(in_text)
       synthesis_input = texttospeech.SynthesisInput(text=in_text)
       response = client.synthesize_speech(
       input=synthesis_input, voice=voice, audio_config=audio_config
       )

       # The response's audio_content is binary.
       f11 = ''.join([str(i+1)+".wav"])
       fname = os.path.join(path,f11)
       with open(fname, "wb") as out:
          # Write the response to the output file.
          out.write(response.audio_content)
          print(['Audio content written to file: '+fname])
       i+=1

ERROR:grpc._plugin_wrapping:AuthMetadataPluginCallback "<google.auth.transport.grpc.AuthMetadataPlugin object at 0x7a88dbe14280>" raised exception!
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/google/auth/compute_engine/credentials.py", line 128, in refresh
    self._retrieve_info(request)
  File "/usr/local/lib/python3.10/dist-packages/google/auth/compute_engine/credentials.py", line 101, in _retrieve_info
    info = _metadata.get_service_account_info(
  File "/usr/local/lib/python3.10/dist-packages/google/auth/compute_engine/_metadata.py", line 323, in get_service_account_info
    return get(request, path, params={"recursive": "true"})
  File "/usr/local/lib/python3.10/dist-packages/google/auth/compute_engine/_metadata.py", line 248, in get
    raise exceptions.TransportError(
google.auth.exceptions.TransportError: ("Failed to retrieve http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/?recursive=true from the Go

It takes a lot of help to finish these


ServiceUnavailable: 503 Getting metadata from plugin failed with error: ("Failed to retrieve http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/?recursive=true from the Google Compute Engine metadata service. Status: 404 Response:\nb''", <google.auth.transport.requests._Response object at 0x7a88ec19b160>)

## Analysis of generated files, get f0 and SR values

Must have the generated folders in the previous section

In [ ]:
import csv
import os
import pandas as pd
import numpy as np
import wave
import string
import contextlib
import jiwer
from huggingsound import SpeechRecognitionModel

def wpm(datapath):
    model = SpeechRecognitionModel("jonatasgrosman/wav2vec2-large-xlsr-53-english")

    audio_paths = [datapath+".wav"]
    transcriptions = model.transcribe(audio_paths)
    print(transcriptions[0]["transcription"])
    trans = transcriptions[0]["transcription"]

    # Count the number of words in the list
    res = sum([i.strip(string.punctuation).isalpha() for i in trans.split()])

    # Get the duration
    start_t = transcriptions[0]["start_timestamps"][0]
    end_t = transcriptions[0]["end_timestamps"][-1]
    duration = end_t - start_t

    # Words per minute
    wpm3 = res*60/(duration/1000)
    dur2 = duration/1000
    print("Duration: ")
    print(dur2)
    print("Words per minute: ")
    print(wpm3)

    return [dur2, wpm3, trans]

file = open('en_US_voices.csv', "r", encoding='utf-8-sig')
v_list = csv.reader(file)
cwd = os.getcwd()

#Open csv containing speech prompts
with open('prompts2.csv', "r", encoding='utf-8-sig') as csv_file2:
    csv_reader2 = csv.reader(csv_file2)
    rows2 = list(csv_reader2)

for row in v_list:

    datapath = './'+''.join(row)+'/'
    f_name = ""

    # open the file in the write mode
    f = open(datapath+'summary_f0.csv', 'w')
    # create the csv writer
    writer = csv.writer(f)
    header = ["File", "Reference", "Transcription", "Min F0", "Median F0", "Max F0", "Duration", "WPM","WER"]
    # write a row to the csv file
    writer.writerow(header)


    for k in range(1,13):

        os.system("crepe "+datapath+f_name+str(k)+".wav --viterbi")
        crepe_vec = pd.read_csv(datapath+f_name+str(k)+'.f0.csv')
        crepe_time = pd.DataFrame(crepe_vec.iloc[:,0].to_numpy())
        crepe_freq = pd.DataFrame(crepe_vec.iloc[:,1].to_numpy())
        crepe_conf = pd.DataFrame(crepe_vec.iloc[:,2].to_numpy())
        threshold = 0.75
        conf_thresh = [1 if i>threshold else 0 for i in crepe_conf.to_numpy()]
        freq_thresh = np.transpose(np.asarray(conf_thresh))*np.asarray(crepe_freq).reshape(-1)
        nz_freq = freq_thresh[np.nonzero(freq_thresh)]

        min_f0 = np.min(nz_freq)
        med_f0 = np.median(nz_freq)
        max_f0 = np.max(nz_freq)
        print("The minimum is: ")
        print(min_f0)
        print("The median is: ")
        print(med_f0)
        print("The maximum is: ")
        print(max_f0)

        [dur, w, tr] = wpm(datapath+f_name+str(k))
        reftxt = ' '.join(rows2[k-1])
        wer = jiwer.wer(reftxt.lower(),tr)
        print("The word error rate is: ")
        print(wer)
        row = [datapath+f_name+str(k), reftxt.lower(), tr, str(min_f0), str(med_f0), str(max_f0), dur, w, wer]
        # write a row to the csv file
        writer.writerow(row)

    # close the file
    f.close()

CREPE: Processing ./en-US-Standard-C/1.wav ... (1/1)
2024-04-11 03:14:57.852156: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-04-11 03:14:57.883677: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-04-11 03:14:58.378371: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2024-04-11 03:15:00.499564: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been bui

8/8 ━━━━━━━━━━━━━━━━━━━━ 5s 587ms/step
CREPE: Saved the estimated frequencies and confidence values at ./en-US-Standard-C/1.f0.csv
The minimum is: 
111.082
The median is: 
176.26600000000002
The maximum is: 
243.278
04/11/2024 03:15:06 - INFO - huggingsound.speech_recognition.model - Loading model...


Some weights of the model checkpoint at jonatasgrosman/wav2vec2-large-xlsr-53-english were not used when initializing Wav2Vec2ForCTC: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'wav2vec2.encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at jonatasgrosman/wav2vec2-large-xlsr-53-english and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
Y

it takes a lot of help to finish these
Duration: 
1.82
Words per minute: 
296.7032967032967
The word error rate is: 
0.0


CREPE: Processing ./en-US-Standard-C/2.wav ... (1/1)
2024-04-11 03:15:09.204212: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-04-11 03:15:09.240168: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-04-11 03:15:09.742364: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2024-04-11 03:15:10.594015: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been bui

...

the house stood on a slight rise just on the edge of the villageit stood on its own and looked over a broadspread of west country farmlandnot a remarkable house by any means it was about thirty years-old squattish suarish made of brick and had four windows set in front of a size and proportion which more or less exactly failed to please the eye
Duration: 
22.32
Words per minute: 
172.04301075268816
The word error rate is: 
0.18571428571428572


CREPE: Processing ./en-US-Standard-B/12.wav ... (1/1)
2024-04-11 03:22:05.199794: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-04-11 03:22:05.233183: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-04-11 03:22:05.779888: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2024-04-11 03:22:07.910912: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been bu

65/65 ━━━━━━━━━━━━━━━━━━━━ 50s 774ms/step
CREPE: Saved the estimated frequencies and confidence values at ./en-US-Standard-B/12.f0.csv
The minimum is: 
94.095
The median is: 
145.038
The maximum is: 
412.391
04/11/2024 03:22:59 - INFO - huggingsound.speech_recognition.model - Loading model...


Some weights of the model checkpoint at jonatasgrosman/wav2vec2-large-xlsr-53-english were not used when initializing Wav2Vec2ForCTC: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'wav2vec2.encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at jonatasgrosman/wav2vec2-large-xlsr-53-english and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
Y

the only person for whom the house was in any way special was arthur dent and that was only because it happened to be the one he lived inhe had lived in it for about three years ever since he had moved out of london because it made him nervous and irritablehe was about thirty as well dark-haired and never quite at ease with himself
Duration: 
19.82
Words per minute: 
193.74369323915238
The word error rate is: 
0.14705882352941177
